In [26]:
from pathlib import Path
import re
import pandas as pd
from pandas import DataFrame

from helpers import rename_patient
from written_thesis.helpers import PRETTY_FEATURE_NAMES_MAP

In [12]:
in_path = Path('~/thesis_files/statistical_results/.model_feature_qualifications.pkl').expanduser()
out_root = Path('~/Developer/MastersThesis/src/tables').expanduser()
cnn_out_path = out_root / 'cnn_feature_qualifications.tex'
ens_out_path = out_root / 'ensemble_feature_qualifications.tex'

In [13]:
orig = pd.read_pickle(in_path)
orig.head()

seizures                   CNN              \
metric                 p_rayleigh_bh significant   roc_auc roc_auc_met   
patient       feature                                                    
competition-1 corrcoef      0.404490       False  0.610688       False   
              acfw_D        0.000561        True  0.610688       False   
              acfw_P        0.008206        True  0.610688       False   
              var_D         0.607431       False  0.610688       False   
              var_P         0.032674        True  0.610688       False   

                                                                              \
metric                  p_rayleigh_bh significant p_perm_bh    met qualified   
patient       feature                                                          
competition-1 corrcoef  2.284097e-106        True  0.973005   True     False   
              acfw_D     1.311392e-54        True  0.001000  False     False   
              acfw_P     1.048648e-01       False  0.006999  False     False   
              var_D      4.328165e-17        True  0.219648   True     False   
              var_P      2.566058e-58        True  0.001000  False     False   

                       ensemble                                         \
metric                  roc_auc roc_auc_met  p_rayleigh_bh significant   
patient       feature                                                    
competition-1 corrcoef  0.59268       False  1.334214e-138        True   
              acfw_D    0.59268       False   5.955082e-81        True   
              acfw_P    0.59268       False   2.067303e-02        True   
              var_D     0.59268       False   2.161849e-19        True   
              var_P     0.59268       False   1.917672e-78        True   

                                                   
metric                 p_perm_bh    met qualified  
patient       feature                              
competition-1 corrcoef  0.984603   True     False  
              acfw_D    0.000750  False     False  
              acfw_P    0.002999  False     False  
              var_D     0.258179   True     False  
              var_P     0.001000  False     False

In [14]:
cnn = orig.drop(columns=['ensemble']).copy()
ens = orig.drop(columns=['CNN']).copy()

In [15]:
cnn_qual = cnn[cnn[('CNN', 'qualified')]]
cnn_qual

seizures                   CNN              \
metric                p_rayleigh_bh significant   roc_auc roc_auc_met   
patient       feature                                                   
competition-2 acfw_D       0.012896        True  0.796782        True   

                                                                            
metric                 p_rayleigh_bh significant p_perm_bh   met qualified  
patient       feature                                                       
competition-2 acfw_D   4.843678e-123        True   0.10138  True      True

In [16]:
ens_qual = ens[ens[('ensemble', 'qualified')]]
ens_qual

seizures              ensemble              \
metric               p_rayleigh_bh significant   roc_auc roc_auc_met   
patient      feature                                                   
U002-DE01-17 Alpha_P      0.003536        True  0.787532        True   
             Beta_P       0.003299        True  0.787532        True   

                                                                          
metric               p_rayleigh_bh significant p_perm_bh   met qualified  
patient      feature                                                      
U002-DE01-17 Alpha_P  4.651312e-11        True  0.243497  True      True  
             Beta_P   4.336590e-07        True  0.613377  True      True

In [45]:
def apply_transformations(df: DataFrame, model: str) -> DataFrame:
    t = df.copy()

    # Abbreviate patients
    idx_df = t.index.to_frame(index=False)
    idx_df['patient'] = idx_df['patient'].map(rename_patient)
    t.index = pd.MultiIndex.from_frame(idx_df)

    t = t.drop(columns=[('seizures', 'significant')] + [(model, c) for c in
                                                        ['significant', 'roc_auc_met', 'met', 'qualified']])

    t = t.rename_axis(index={'patient': 'Patient', 'feature': 'Feature'},
                      columns={'metric': None})
    t = t.rename(
        columns={
            'seizures': 'Seizures',
            'ensemble': 'Model',
            'CNN': 'Model',

            'p_rayleigh_bh': r'$p_{\mathrm{Rayleigh, BH}}$',
            'significant': '$<0.05$',

            'roc_auc': 'ROC AUC',
            'roc_auc_met': r'$\geq 0.75$',

            'p_perm_bh': r'$p_{\mathrm{Permutation, BH}}$',
            'met': r'$\geq 0.05$',
        },
        index=PRETTY_FEATURE_NAMES_MAP,
    )
    t.insert(t.columns.get_loc(('Model', 'ROC AUC')), ('Model', 'Type'), model)

    return t


cnn_t = apply_transformations(cnn_qual, 'CNN')
ens_t = apply_transformations(ens_qual, 'ensemble')

combined = pd.concat([cnn_t, ens_t])
combined

FrozenList([None, None])

In [43]:
l = combined.to_latex(float_format='%.3f', escape=False, multicolumn=True, multicolumn_format='c', column_format='ll|r|rrrr')
l = l.replace(r'\bottomrule'+'\n', '')
l = re.sub(r'(toprule|midrule|bottomrule)', 'hline', l)
print(l)


\begin{tabular}{ll|r|rrrr}
\hline
 &  & Seizures & \multicolumn{4}{|c}{Model} \\
 &  & $p_{\mathrm{Rayleigh, BH}}$ & Type & ROC AUC & $p_{\mathrm{Rayleigh, BH}}$ & $p_{\mathrm{Permutation, BH}}$ \\
Patient & Feature &  &  &  &  &  \\
\hline
C02 & ACFW (D) & 0.013 & CNN & 0.797 & 0.000 & 0.101 \\
\cline{1-7}
\multirow[t]{2}{*}{U17} & Alpha (P) & 0.004 & ensemble & 0.788 & 0.000 & 0.243 \\
 & Beta (P) & 0.003 & ensemble & 0.788 & 0.000 & 0.613 \\
\cline{1-7}
\end{tabular}

